# Day 084 — Solution: A Researcher + Writer Duo

In [ ]:
_SRC = '"""multi_agent.py — Day 084: Multi-Agent Systems.\n\nDays 79-83 gave one agent tools, reasoning, memory, and planning. Today two\nagents collaborate: a ResearcherAgent gathers facts, then hands off to a\nWriterAgent that turns them into a polished document. An Orchestrator owns both,\nruns the pipeline, and keeps a record of every Handoff between agents.\n\nPieces:\n  safe_parse_json / call_llm  - reused (Day 79)\n  build_researcher_prompt / ResearcherAgent   - specialist researcher\n  build_writer_prompt / WriterAgent           - specialist writer\n  Handoff                                     - explicit data transfer between agents\n  summarize_handoffs                          - render the handoff chain as text\n  run_duo                                     - chain researcher -> writer\n  Orchestrator                                - owns both agents; runs and records\n\nSetup:\n    pip install ollama\n    ollama pull llama3.2\n"""\nimport json\n\n# ── helpers reused from Day 79 ───────────────────────────────────────────────\ndef safe_parse_json(text):\n    """Slice first \'{\' to last \'}\' and parse. Returns dict|None (Day 79)."""\n    start, end = text.find("{"), text.rfind("}")\n    if start == -1 or end == -1 or end < start:\n        return None\n    try:\n        data = json.loads(text[start:end + 1])\n    except (json.JSONDecodeError, ValueError):\n        return None\n    return data if isinstance(data, dict) else None\n\n\ndef call_llm(messages, llm_fn=None):\n    """Call the chat model, or the injected llm_fn(messages) -> str (Day 79)."""\n    if llm_fn is not None:\n        return llm_fn(messages)\n    import ollama\n    resp = ollama.chat(model="llama3.2", messages=messages)\n    return resp["message"]["content"]\n\n# ── researcher specialist ─────────────────────────────────────────────────────\ndef build_researcher_prompt(query):\n    """Build a prompt for the researcher role: gather facts on a topic."""\n    system = "\\n".join([\n        "You are a research specialist. Your job is to gather relevant facts and",\n        "key information about the topic given to you.",\n        "",\n        "Return a structured list of the most important findings.",\n        "Be factual, concise, and cover the main points.",\n    ])\n    return [{"role": "system", "content": system},\n            {"role": "user", "content": "Research topic: " + str(query)}]\n\n\nclass ResearcherAgent:\n    """A specialist that researches a topic and returns structured findings.\n\n    Each call to research() returns a string of findings and records the\n    exchange in history. The agent has one job: gather facts. It passes its\n    output to the next agent via a Handoff — it does not write, review, or\n    plan.\n\n    Example::\n\n        researcher = ResearcherAgent(llm_fn=my_llm_fn)\n        findings = researcher.research("topological sort algorithms")\n    """\n\n    def __init__(self, llm_fn=None):\n        self._llm_fn = llm_fn\n        self._history = []\n\n    def research(self, query):\n        """Research a query and return findings as a string."""\n        messages = build_researcher_prompt(query)\n        findings = call_llm(messages, llm_fn=self._llm_fn)\n        self._history.append({"query": query, "findings": findings})\n        return findings\n\n    def history(self):\n        """Return a copy of the research history."""\n        return list(self._history)\n\n    def clear_history(self):\n        """Clear the history in place."""\n        self._history.clear()\n\n# ── writer specialist ─────────────────────────────────────────────────────────\ndef build_writer_prompt(findings, style="concise", instructions=None):\n    """Build a prompt for the writer role: turn findings into a document."""\n    system_parts = [\n        "You are a writing specialist. Turn the provided findings into",\n        "a polished, well-structured document.",\n        "Style: " + str(style) + ".",\n    ]\n    if instructions:\n        system_parts.append("Additional instructions: " + str(instructions))\n    system = "\\n".join(system_parts)\n    user = "Findings:\\n" + str(findings)\n    return [{"role": "system", "content": system},\n            {"role": "user", "content": user}]\n\n\nclass WriterAgent:\n    """A specialist that turns research findings into a polished document.\n\n    The writer has one job: take findings (a string from the ResearcherAgent\n    or any other source) and produce a well-structured document. The style\n    controls the tone (e.g. \'concise\', \'detailed\', \'formal\').\n\n    Example::\n\n        writer = WriterAgent(llm_fn=my_llm_fn, style="concise")\n        document = writer.write(findings)\n    """\n\n    def __init__(self, llm_fn=None, style="concise"):\n        self._llm_fn = llm_fn\n        self.style = style\n        self._history = []\n\n    def write(self, findings, instructions=None):\n        """Write a document from findings. Returns the document string."""\n        messages = build_writer_prompt(findings, style=self.style,\n                                       instructions=instructions)\n        document = call_llm(messages, llm_fn=self._llm_fn)\n        self._history.append({"findings": findings, "document": document})\n        return document\n\n    def history(self):\n        """Return a copy of the writing history."""\n        return list(self._history)\n\n    def clear_history(self):\n        """Clear the history in place."""\n        self._history.clear()\n\n# ── handoffs: explicit data passing between agents ────────────────────────────\nfrom dataclasses import dataclass, field\n\n\n@dataclass\nclass Handoff:\n    """An explicit record of data passed from one agent to the next.\n\n    Attributes:\n        from_agent: name of the sending agent (\'researcher\', \'writer\', ...).\n        to_agent:   name of the receiving agent.\n        content:    the data being passed (findings string, document, ...).\n        metadata:   optional dict for any extra context (topic, style, ...).\n    """\n    from_agent: str\n    to_agent: str\n    content: str\n    metadata: dict = field(default_factory=dict)\n\n\ndef summarize_handoffs(handoffs):\n    """Render the handoff chain as a human-readable text for debugging.\n\n    Each line shows the sender, receiver, and the first 80 characters of the\n    content, so you can see the full flow at a glance.\n    """\n    lines = []\n    for h in handoffs:\n        preview = h.content[:80].replace("\\n", " ")\n        lines.append(h.from_agent + " -> " + h.to_agent + ": " + preview)\n    return "\\n".join(lines)\n\n# ── the researcher-writer pipeline ───────────────────────────────────────────\ndef run_duo(task, researcher, writer):\n    """Chain a ResearcherAgent then a WriterAgent for one task.\n\n    The researcher gathers findings; a Handoff carries them to the writer;\n    the writer produces the document; a second Handoff records the output.\n\n    Returns {"findings": str, "document": str, "handoffs": list[Handoff]}.\n    """\n    findings = researcher.research(task)\n    h1 = Handoff("researcher", "writer", findings, metadata={"task": str(task)})\n    document = writer.write(findings)\n    h2 = Handoff("writer", "user", document, metadata={"task": str(task)})\n    return {"findings": findings, "document": document, "handoffs": [h1, h2]}\n\n# ── the orchestrator ──────────────────────────────────────────────────────────\nclass Orchestrator:\n    """Owns specialist agents; coordinates the researcher -> writer pipeline.\n\n    An Orchestrator is one object that owns a ResearcherAgent and a WriterAgent,\n    runs them in sequence for each task, and keeps a record of every run and\n    every Handoff. Give it a task, get back findings, a document, and the full\n    handoff trail.\n\n    Example::\n\n        orch = Orchestrator(llm_fn=my_llm_fn)\n        result = orch.run("What is the CAP theorem?")\n        print(result["document"])\n        print(summarize_handoffs(orch.handoffs()))\n    """\n\n    def __init__(self, llm_fn=None, style="concise"):\n        self.researcher = ResearcherAgent(llm_fn=llm_fn)\n        self.writer = WriterAgent(llm_fn=llm_fn, style=style)\n        self._runs = []\n\n    def run(self, task):\n        """Research then write. Returns {"findings","document","handoffs"}."""\n        result = run_duo(task, self.researcher, self.writer)\n        self._runs.append({"task": task, **result})\n        return result\n\n    def handoffs(self):\n        """All Handoff objects across every run, in order."""\n        all_h = []\n        for r in self._runs:\n            all_h.extend(r.get("handoffs", []))\n        return all_h\n\n    def history(self):\n        """Return a copy of the run history."""\n        return list(self._runs)\n\n    def clear_history(self):\n        """Clear history and per-agent histories in place."""\n        self._runs.clear()\n        self.researcher.clear_history()\n        self.writer.clear_history()\n'
from pathlib import Path
Path('multi_agent.py').write_text(_SRC, encoding='utf-8')
print('multi_agent.py written.')

In [ ]:

from multi_agent import (
    ResearcherAgent, WriterAgent, Handoff, summarize_handoffs,
    run_duo, Orchestrator,
)

def _mock_multi(findings='Fact: X.', document='Doc: done.'):
    def _fn(messages):
        system = messages[0]['content'] if messages else ''
        return findings if 'research specialist' in system.lower() else document
    return _fn

# 1. ResearcherAgent
r = ResearcherAgent(llm_fn=_mock_multi(findings='Found it.'))
out = r.research('AI')
assert out == 'Found it.' and len(r.history()) == 1
r.history().clear(); assert len(r.history()) == 1   # copy
r.clear_history(); assert len(r.history()) == 0
print("✅ ResearcherAgent (research / history / clear_history)")

# 2. WriterAgent
w = WriterAgent(llm_fn=_mock_multi(document='Written.'), style='formal')
doc = w.write('some findings')
assert doc == 'Written.' and 'formal' in str(w.style)
assert len(w.history()) == 1
print("✅ WriterAgent (write / style / history)")

# 3. Handoff + summarize
h1 = Handoff('researcher', 'writer', 'the facts')
assert h1.metadata == {}
h2 = Handoff('a', 'b', 'x' * 200)
summary = summarize_handoffs([h1, h2])
assert 'researcher -> writer' in summary
lines = summary.splitlines()
assert len(lines[1]) < 200          # preview truncated
assert summarize_handoffs([]) == ''
print("✅ Handoff / summarize_handoffs")

# 4. run_duo
researcher = ResearcherAgent(llm_fn=_mock_multi(findings='Key fact.'))
writer = WriterAgent(llm_fn=_mock_multi(document='Final doc.'))
result = run_duo('test topic', researcher, writer)
assert result['findings'] == 'Key fact.' and result['document'] == 'Final doc.'
assert len(result['handoffs']) == 2
h_r, h_w = result['handoffs']
assert h_r.from_agent == 'researcher' and h_r.to_agent == 'writer'
assert h_w.from_agent == 'writer' and h_w.to_agent == 'user'
print("✅ run_duo (findings / document / handoffs)")

# 5. Orchestrator
orch = Orchestrator(llm_fn=_mock_multi(findings='Fact A.', document='Doc A.'))
res = orch.run('first task')
assert res['findings'] == 'Fact A.' and res['document'] == 'Doc A.'
orch.run('second task')
assert len(orch.history()) == 2
orch.history().clear(); assert len(orch.history()) == 2   # copy
assert len(orch.handoffs()) == 4     # 2 runs × 2 handoffs
orch.clear_history()
assert len(orch.history()) == 0 and len(orch.handoffs()) == 0
print("✅ Orchestrator (run / handoffs / history / clear_history)")

print("\nMulti-agent system complete!")
